For the outcome prediction task, the selected clinical endpoint is Recurrence-Free Survival (RFS), defined as time without any recurrence, censoring all others, including deaths. In particular, local, regional, and distant metastases are events and all others are censored. Time to event, defined in days, starts with the end of radiation therapy.

https://hecktor25.grand-challenge.org/ground-truth/

In [ ]:
import pickle
with open(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\pickle datasets\hecktor.pickle", "rb") as f:
    patients = pickle.load(f)

In [ ]:
print(len(patients))

In [ ]:
p = patients[list(patients.keys())[0]]
print(p.id)
print(p.center_id)

In [ ]:
# RFS in days transformed into binary 2y RFS
rfs = p.clinical['RFS']
if not(rfs is None):
    if rfs < 365*2:
        0
    else:
        1

In [ ]:
import pandas

# HECKTOR 
df = pandas.read_csv(r"E:\bilel\HECKTOR 2025 Training Data\Task 3\HECKTOR_2025_Training_Task_3.csv")
print(df.columns)

hpv = df["HPV Status"]
hpv[hpv == 1] = "positive"
hpv[hpv == 0] = "negative"

df = pandas.read_csv(r"E:\bilel\HECKTOR 2025 Training Data\Task 2\HECKTOR_2025_Training_Task_2.csv")
print(df.columns)

# "In particular, local, regional, and distant metastases are events and all others are censored. 
# Time to event, defined in days, starts with the end of radiation therapy."
lrr = df["Relapse"]
os = df["RFS"]

plot_clinical(hpv=hpv, lrr=lrr, os=os)

# plot data statistics

In [ ]:
list(patients.values())[0].clinical

In [ ]:
import pandas
import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_recc_data(data):
    rfs2y = []
    for i in data:
        dt, cens = i["dt"], i["censored"]
        if not(cens):
            if dt < 2*365:
                rfs2y.append({"y": "yes", "year": 2})
                rfs2y.append({"y": "yes", "year": 5})
            elif 2*365 < dt and dt < 5*365:
                rfs2y.append({"y": "no", "year": 2})
                rfs2y.append({"y": "yes", "year": 5})
            else:
                rfs2y.append({"y": "no", "year": 2})
                rfs2y.append({"y": "no", "year": 5})
        else:
            if dt > 2*365:
                rfs2y.append({"y": "no", "year": 2})
            if dt > 5*365:
                rfs2y.append({"y": "no", "year": 5})
    rfs2y = pandas.DataFrame(rfs2y)

    fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]])
    for c, year in enumerate(rfs2y["year"].unique()):
        df = rfs2y[rfs2y["year"] == year]
        print(len(df))
        labels = df["y"].unique()
        values = [len(df[df["y"] == i]) for i in labels]
        fig.add_trace(go.Pie(labels=labels, values=values, title=f"{year} years"), 1, c+1)
    
    fig.update_layout(autosize=False, width=800, height=400)
    fig.show()


data = []
for p in patients.values():
    try:
        if p.clinical['Relapse']:
            data.append({"dt": int(p.clinical["RFS"]), "censored": False})
        else:
            data.append({"dt": int(p.clinical["RFS"]), "censored": True})
    except KeyError:
        continue

plot_recc_data(data)

# center ID

In [ ]:
# HECKTOR Challenge 2025 (10)
#   HGJ: Hôpital général juif
#   CHUS: Centre hospitalier universitaire de Sherbooke
#   HMR: Hôpital Maisonneuve-Rosemont
#   CHUM: Centre hospitalier de l’Université de Montréal
#   CHUP: Centre Hospitalier Universitaire de Poitiers
#   MDA: MD Anderson Cancer Center
#   USZ: UniversitätsSpital Zürich
#   CHB: Centre Henri Becquerel
#   CHUB: Centre Hospitalier Universitaire de Brest
#   CHUN: Centre Hospitalier Universitaire de Nantes

# generate clinical data

In [ ]:
col = {
    "PatientID": "PatientID",
    "Age": "Age",
    "Gender": "Gender",
    "Tobacco Consumption": "Tobacco Consumption",
    "Alcohol Consumption": "Alcohol Consumption",
    "Performance Status": "Performance Status",
    "M-stage": "M-stage",
    "HPV Status": "HPV Status",
}

df1 = pandas.read_csv(r"E:\bilel\HECKTOR 2025 Training Data\Task 1\HECKTOR_2025_Training_Task_1.csv")
df2 = pandas.read_csv(r"E:\bilel\HECKTOR 2025 Training Data\Task 2\HECKTOR_2025_Training_Task_2.csv")
df3 = pandas.read_csv(r"E:\bilel\HECKTOR 2025 Training Data\Task 3\HECKTOR_2025_Training_Task_3.csv")

df = df1.merge(df2, how='outer').merge(df3, how='outer')
df = df[[c for c in df.columns if c in col.keys()]]
df = df.rename(columns=col, errors="raise")
df.to_csv(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\dfHECKTOR 2025.csv", index=False)